# 01 — SQL Exploration

Query the hostpathogen database with SQL + pandas.

In [ ]:
from hostpathogen.data.loader import query, to_df, get_connection

## Basic SELECT

In [ ]:
rows = query("SELECT name, gram_stain, strategy FROM pathogens")
for r in rows:
    print(f"{r['name']:30s} {r['gram_stain']:15s} {r['strategy']}")

## JOIN: effectors + pathogens

In [ ]:
df = to_df("""
    SELECT p.name AS pathogen, e.name AS effector, e.type, e.host_target
    FROM effectors e
    JOIN pathogens p ON e.pathogen_id = p.id
    ORDER BY p.name, e.name
""")
df

## Aggregation: most-targeted host proteins

In [ ]:
df = to_df("""
    SELECT h.name, COUNT(*) AS n_effectors
    FROM effector_targets et
    JOIN host_proteins h ON et.host_protein_id = h.id
    GROUP BY h.name
    ORDER BY n_effectors DESC
    LIMIT 10
""")
df

## Filtering by pathogen strategy

In [ ]:
df = to_df("""
    SELECT p.name AS pathogen, e.name AS effector, h.name AS host_target
    FROM effector_targets et
    JOIN effectors e ON et.effector_id = e.id
    JOIN host_proteins h ON et.host_protein_id = h.id
    JOIN pathogens p ON e.pathogen_id = p.id
    WHERE p.strategy = 'escape'
""")
df

## Subquery: effectors targeting Rab-family proteins

In [ ]:
df = to_df("""
    SELECT e.name AS effector, h.name AS host_target
    FROM effector_targets et
    JOIN effectors e ON et.effector_id = e.id
    JOIN host_proteins h ON et.host_protein_id = h.id
    WHERE h.name LIKE 'Rab%'
    ORDER BY h.name, e.name
""")
df

## Phagosome stage markers: what's present at each stage?

In [ ]:
df = to_df("""
    SELECT ms.name AS stage, hp.name AS protein, sm.presence
    FROM stage_markers sm
    JOIN maturation_stages ms ON sm.stage_id = ms.id
    JOIN host_proteins hp ON sm.host_protein_id = hp.id
    WHERE sm.presence = 1
    ORDER BY ms.stage_order, hp.name
""")
df